In [1]:
import os
import numpy as np 
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModel
from tqdm import tqdm
from scipy.stats import spearmanr

In [2]:
# Configuration
AVA_IMAGES_DIR = "/Users/xinghebai/Desktop/Master Project/Teacher Baseline AVA/images"
AVA_ANNOTATIONS_FILE = "/Users/xinghebai/Desktop/Master Project/Teacher Baseline AVA/AVA.txt"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16  # small for testing
SUBSET_SIZE = 500  # use 500 images for initial test

In [3]:
def load_ava_annotations(txt_file, subset_size=None):
    data = []
    with open(txt_file, "r") as f:
        for line in f:
            row = line.strip().split()
            img_id = row[1]
            ratings = list(map(int, row[2:12]))
            total_votes = sum(ratings)
            mean_score = sum((i+1)*count for i, count in enumerate(ratings)) / total_votes if total_votes > 0 else 0.0
            data.append((img_id, mean_score))
    if subset_size:
        data = data[:subset_size]
    return data

ava_data = load_ava_annotations(AVA_ANNOTATIONS_FILE, SUBSET_SIZE)
print(f"Loaded {len(ava_data)} annotations")
print("Example:", ava_data[0])


Loaded 500 annotations
Example: ('953619', 5.637096774193548)


In [4]:
class AVADataset(Dataset):
    def __init__(self, data, img_dir, transform=None):
        self.data = data
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_id, score = self.data[idx]
        img_path = os.path.join(self.img_dir, f"{img_id}.jpg")
        if not os.path.exists(img_path):
            img = Image.new("RGB", (224, 224), (0,0,0))  # fallback
        else:
            img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(score, dtype=torch.float32)


In [5]:
print("Loading models...")

# SigLIP
siglip_processor = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")
siglip_model = AutoModel.from_pretrained("google/siglip-base-patch16-224").to(DEVICE)
siglip_model.eval()
siglip_head = torch.nn.Linear(siglip_model.config.vision_config.hidden_size, 1).to(DEVICE)
siglip_head.eval()

# CLIP
clip_processor = AutoProcessor.from_pretrained("openai/clip-vit-large-patch14")
clip_model = AutoModel.from_pretrained("openai/clip-vit-large-patch14").to(DEVICE)
clip_model.eval()
clip_head = torch.nn.Linear(clip_model.config.projection_dim, 1).to(DEVICE)
clip_head.eval()


Loading models...


Linear(in_features=768, out_features=1, bias=True)

In [6]:
print(f"SigLIP head expects input size: {siglip_head.in_features}")
print(f"CLIP head expects input size: {clip_head.in_features}")

SigLIP head expects input size: 768
CLIP head expects input size: 768


In [7]:
def siglip_transform(image):
    return siglip_processor(images=image, return_tensors="pt")['pixel_values'].squeeze(0)

def clip_transform(image):
    return clip_processor(images=image, return_tensors="pt")['pixel_values'].squeeze(0)


In [8]:
siglip_dataset = AVADataset(ava_data, AVA_IMAGES_DIR, transform=siglip_transform)
clip_dataset   = AVADataset(ava_data, AVA_IMAGES_DIR, transform=clip_transform)

siglip_loader = DataLoader(siglip_dataset, batch_size=BATCH_SIZE, shuffle=False)
clip_loader   = DataLoader(clip_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [9]:
@torch.no_grad()
def evaluate_model(model, head, loader, device):
    predictions, ground_truth = [], []
    for images, labels in tqdm(loader):
        images = images.to(device)
        labels = labels.to(device)
        feats = model.get_image_features(pixel_values=images)
        # feats should be [batch, hidden_size], no need to flatten if already 2D
        if feats.ndim > 2:
            feats = feats.reshape(feats.shape[0], -1)
        preds = head(feats).squeeze(-1).cpu().numpy()
        predictions.extend(preds.tolist())
        ground_truth.extend(labels.cpu().numpy().tolist())
    return predictions, ground_truth


In [10]:
print("Evaluating SigLIP...")
siglip_preds, gt = evaluate_model(siglip_model, siglip_head, siglip_loader, DEVICE)

print("Evaluating CLIP...")
clip_preds, _ = evaluate_model(clip_model, clip_head, clip_loader, DEVICE)

# Compute SRCC
srcc_siglip, _ = spearmanr(gt, siglip_preds)
srcc_clip, _ = spearmanr(gt, clip_preds)

print(f"Subset SRCC: SigLIP={srcc_siglip:.4f}, CLIP={srcc_clip:.4f}")


Evaluating SigLIP...


100%|███████████████████████████████████████████| 32/32 [00:28<00:00,  1.14it/s]


Evaluating CLIP...


100%|███████████████████████████████████████████| 32/32 [01:27<00:00,  2.74s/it]

Subset SRCC: SigLIP=0.0346, CLIP=-0.0664


In [11]:
import torch.nn as nn
import torch.optim as optim
import random

# Configuration
EPOCHS = 5
LEARNING_RATE = 1e-4
TRAIN_SPLIT_RATIO = 0.8
SUBSET_SIZE = 2000 

# Using the full dataset for training and evaluation
ava_data = load_ava_annotations(AVA_ANNOTATIONS_FILE, SUBSET_SIZE)
random.shuffle(ava_data)
split_index = int(len(ava_data) * TRAIN_SPLIT_RATIO)
train_data = ava_data[:split_index]
val_data = ava_data[split_index:]

print(f"Total samples: {len(ava_data)}")
print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

Total samples: 2000
Training samples: 1600
Validation samples: 400


In [12]:
def train_one_epoch(model, head, loader, criterion, optimizer, device):
    """Runs a single training epoch."""
    model.eval()  # Keep the base model in eval mode
    head.train()  # Set the prediction head to train mode

    running_loss = 0.0
    for images, labels in tqdm(loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        with torch.no_grad():
            features = model.get_image_features(pixel_values=images)
        
        outputs = head(features).squeeze(-1)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)

def evaluate(model, head, loader, criterion, device):
    """Evaluates the model on the validation set."""
    model.eval()
    head.eval()

    val_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating"):
            images, labels = images.to(device), labels.to(device)

            features = model.get_image_features(pixel_values=images)
            outputs = head(features).squeeze(-1)
            
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            all_preds.extend(outputs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    # Calculate SRCC
    srcc, _ = spearmanr(all_labels, all_preds)

    return val_loss / len(loader), srcc

In [13]:
def train_model(base_model, head, train_loader, val_loader, device):
    """Manages the full training process for one model."""
    
    # Freeze the base model 
    for param in base_model.parameters():
        param.requires_grad = False

    # Use Mean Squared Error for this regression task
    criterion = nn.MSELoss()
    optimizer = optim.Adam(head.parameters(), lr=LEARNING_RATE)

    print(f"\n--- Starting Training for {base_model.config._name_or_path} ---")
    for epoch in range(EPOCHS):
        train_loss = train_one_epoch(base_model, head, train_loader, criterion, optimizer, device)
        val_loss, val_srcc = evaluate(base_model, head, val_loader, criterion, device)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val SRCC: {val_srcc:.4f}")

# Train SigLIP 
siglip_train_dataset = AVADataset(train_data, AVA_IMAGES_DIR, transform=siglip_transform)
siglip_val_dataset = AVADataset(val_data, AVA_IMAGES_DIR, transform=siglip_transform)
siglip_train_loader = DataLoader(siglip_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
siglip_val_loader = DataLoader(siglip_val_dataset, batch_size=BATCH_SIZE)

# Train the SigLIP model's head
train_model(siglip_model, siglip_head, siglip_train_loader, siglip_val_loader, DEVICE)


# Train CLIP 
clip_train_dataset = AVADataset(train_data, AVA_IMAGES_DIR, transform=clip_transform)
clip_val_dataset = AVADataset(val_data, AVA_IMAGES_DIR, transform=clip_transform)
clip_train_loader = DataLoader(clip_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
clip_val_loader = DataLoader(clip_val_dataset, batch_size=BATCH_SIZE)

# Train the CLIP model's head
train_model(clip_model, clip_head, clip_train_loader, clip_val_loader, DEVICE)


--- Starting Training for google/siglip-base-patch16-224 ---


Validating: 100%|███████████████████████████████| 25/25 [00:23<00:00,  1.09it/s]


Epoch 1/5 | Train Loss: 22.4667 | Val Loss: 18.8946 | Val SRCC: 0.1201


Validating: 100%|███████████████████████████████| 25/25 [00:22<00:00,  1.09it/s]


Epoch 2/5 | Train Loss: 15.9961 | Val Loss: 13.3549 | Val SRCC: 0.1516


...

Epoch 3/5 | Train Loss: 9.0079 | Val Loss: 6.8973 | Val SRCC: -0.0050


Validating: 100%|███████████████████████████████| 25/25 [01:09<00:00,  2.80s/it]


Epoch 4/5 | Train Loss: 5.2008 | Val Loss: 4.0837 | Val SRCC: 0.0098


Validating: 100%|███████████████████████████████| 25/25 [01:09<00:00,  2.80s/it]

Epoch 5/5 | Train Loss: 3.0560 | Val Loss: 2.5618 | Val SRCC: 0.0232
